# LeetCode 225 — Implement Stack using Queues bằng Python

**Link đề:** https://leetcode.com/problems/implement-stack-using-queues/
**Độ khó:** Easy · **Chủ đề:** Stack, Queue, Design (thiết kế lớp)

```python
class MyStack:
    def __init__(self):
        ...
    def push(self, x: int) -> None:
        ...
    def pop(self) -> int:
        ...
    def top(self) -> int:
        ...
    def empty(self) -> bool:
        ...
```

## Mục tiêu học
1. Phân biệt **stack** (LIFO — vào sau ra trước) và **queue** (FIFO — vào trước ra trước).
2. Dùng `collections.deque` như một queue **chỉ với thao tác chuẩn**: thêm cuối, xem/lấy đầu, kích thước, rỗng.
3. Biết cách viết một **class thiết kế**: `__init__`, thuộc tính `self.q`, các method.
4. Học 3 thiết kế: hai queue (pop tốn `O(n)`) → hai queue (push tốn `O(n)`) → **một queue xoay vòng** (follow-up).
5. Trace trạng thái queue sau mỗi lệnh, chứng minh invariant, phân tích chi phí từng thao tác, và mô phỏng cách LeetCode gọi class.

## 1. Đề bài

Cài đặt một **stack** (ngăn xếp, LIFO) chỉ bằng **queue** (hàng đợi). Class `MyStack` cần:

| Method | Ý nghĩa |
|---|---|
| `push(x)` | đẩy `x` lên **đỉnh** stack |
| `pop()` | lấy ra và trả về phần tử ở đỉnh |
| `top()` | trả về phần tử ở đỉnh (không lấy ra) |
| `empty()` | `True` nếu stack rỗng |

**Ràng buộc quan trọng:** chỉ được dùng **thao tác chuẩn của queue**: thêm vào **cuối** (push to back), xem/lấy ở **đầu** (peek/pop from front), `size`, `is empty`. Trong Python được mô phỏng queue bằng `list` hoặc `deque`, miễn là chỉ dùng các thao tác đó.

**Ví dụ:**
```text
Input : ["MyStack", "push", "push", "top", "pop", "empty"]
        [[],        [1],    [2],    [],    [],    []]
Output: [null,      null,   null,   2,     2,     false]
```

**Constraints:** `1 <= x <= 9`; tối đa 100 lời gọi; mọi lời gọi `pop`, `top` đều hợp lệ (stack không rỗng).

**Follow-up:** Có thể dùng **chỉ một queue** không?

## 2. Nền tảng: stack và queue khác nhau thế nào?

Hình dung:
- **Stack** = chồng đĩa: đặt đĩa lên trên cùng, lấy đĩa trên cùng. Vào sau **ra trước** (LIFO — Last In First Out).
- **Queue** = hàng người xếp hàng: người đến sau đứng cuối, người đầu hàng được phục vụ trước. Vào trước **ra trước** (FIFO — First In First Out).

Cùng đưa vào `1, 2, 3`:
- Stack lấy ra: `3, 2, 1`.
- Queue lấy ra: `1, 2, 3`.

Thử thách của bài: dùng "hàng người" để giả lập "chồng đĩa".

In [1]:
# Stack bằng list Python (chỉ để so sánh — bài này KHÔNG được dùng cách này)
stack = []
for x in [1, 2, 3]:
    stack.append(x)            # đẩy lên đỉnh
print("stack sau 3 lần append:", stack, "(đỉnh ở cuối list)")
print("pop 3 lần ->", stack.pop(), stack.pop(), stack.pop(), "| còn:", stack)

stack sau 3 lần append: [1, 2, 3] (đỉnh ở cuối list)
pop 3 lần -> 3 2 1 | còn: []


### 2.1. `collections.deque` làm queue

`deque` (double-ended queue) thêm/bớt ở **hai đầu** đều `O(1)`. Ta chỉ dùng phần "queue chuẩn" của nó:

| Thao tác queue chuẩn | Với `deque` | Chi phí |
|---|---|---|
| push to back | `q.append(x)` | `O(1)` |
| pop from front | `q.popleft()` | `O(1)` |
| peek front | `q[0]` | `O(1)` |
| size | `len(q)` | `O(1)` |
| is empty | `not q` hoặc `len(q) == 0` | `O(1)` |

**Không được dùng** (vì không phải thao tác queue): `q.pop()` (lấy ở cuối), `q[-1]` (xem cuối), `q.appendleft()`, `q.rotate()`.

Vì sao không dùng `list.pop(0)` làm "pop from front"? Được phép về luật, nhưng `list.pop(0)` phải dịch mọi phần tử còn lại → `O(n)`. `deque.popleft()` là `O(1)`.

In [2]:
from collections import deque

q = deque()
for x in [1, 2, 3]:
    q.append(x)                 # vào cuối hàng
print("queue:", q, "| đầu hàng q[0] =", q[0], "| len =", len(q))
print("popleft ->", q.popleft(), "| còn:", q)
print("rỗng?", not q, "| bool(deque()) =", bool(deque()))

queue: deque([1, 2, 3]) | đầu hàng q[0] = 1 | len = 3
popleft -> 1 | còn: deque([2, 3])
rỗng? False | bool(deque()) = False


### 2.2. Cú pháp class: `__init__`, `self`, thuộc tính

- `class MyStack:` định nghĩa một kiểu dữ liệu mới.
- `__init__(self)` chạy khi tạo đối tượng `MyStack()`; ta tạo queue ở đây và gắn vào `self.q` để các method khác dùng chung.
- Mỗi method nhận `self` là chính đối tượng đang được gọi.

Ví dụ nhỏ không liên quan bài:

In [3]:
class Counter2:
    def __init__(self):
        self.value = 0          # thuộc tính của đối tượng

    def inc(self) -> None:
        self.value += 1

    def get(self) -> int:
        return self.value

c1, c2 = Counter2(), Counter2()
c1.inc(); c1.inc(); c2.inc()
print("c1 =", c1.get(), "| c2 =", c2.get(), "(mỗi đối tượng có trạng thái riêng)")

c1 = 2 | c2 = 1 (mỗi đối tượng có trạng thái riêng)


## 3. Vấn đề cốt lõi

Queue chỉ cho lấy **đầu**, nhưng stack cần lấy phần tử **mới nhất** — nằm ở **cuối** queue.

```text
push 1, 2, 3 vào queue:   đầu [1, 2, 3] cuối
stack muốn pop:  3   (đang ở cuối!)
```

Hai hướng:
- **Hướng A — "lười khi push, chăm khi pop":** push bình thường vào cuối; khi cần pop, chuyển `n - 1` phần tử đầu sang chỗ khác để lộ phần tử cuối.
- **Hướng B — "chăm khi push, lười khi pop":** mỗi lần push, sắp xếp lại để phần tử mới luôn nằm ở **đầu** queue; pop/top chỉ việc lấy/xem đầu.

## 4. Cách 1 — Hai queue, push `O(1)`, pop `O(n)` (hướng A)

- `q1` chứa dữ liệu theo thứ tự vào; phần tử đỉnh stack là phần tử **cuối** của `q1`.
- `push(x)`: `q1.append(x)`.
- `pop()`: chuyển `len(q1) - 1` phần tử từ đầu `q1` sang `q2`; phần tử còn lại trong `q1` chính là đỉnh → lấy ra. Rồi **đổi vai** `q1, q2 = q2, q1`.
- `top()`: giống `pop` nhưng phải đặt phần tử đỉnh trở lại. Để đơn giản và vẫn đúng luật, ta có thể lưu thêm biến `self._top` cập nhật ở mỗi `push`/`pop`.

In [4]:
class MyStackTwoQueuesLazy:
    def __init__(self):
        self.q1 = deque()
        self.q2 = deque()
        self._top = None             # giá trị đỉnh hiện tại

    def push(self, x: int) -> None:
        self.q1.append(x)
        self._top = x

    def pop(self) -> int:
        while len(self.q1) > 1:      # chuyển n-1 phần tử sang q2
            self._top = self.q1.popleft()
            self.q2.append(self._top)
        result = self.q1.popleft()   # phần tử cuối cùng = đỉnh stack
        self.q1, self.q2 = self.q2, self.q1
        if not self.q1:
            self._top = None
        return result

    def top(self) -> int:
        return self._top

    def empty(self) -> bool:
        return not self.q1

s = MyStackTwoQueuesLazy()
s.push(1); s.push(2); s.push(3)
print("top =", s.top(), "| pop =", s.pop(), "| top =", s.top(), "| pop =", s.pop(), "| pop =", s.pop(), "| empty =", s.empty())

top = 3 | pop = 3 | top = 2 | pop = 2 | pop = 1 | empty = True


**Vì sao `_top` sau khi pop đúng?** Trong vòng chuyển, phần tử cuối cùng được chuyển sang `q2` là phần tử đứng **ngay trước** đỉnh cũ — tức đỉnh mới. Ta gán `_top` mỗi lần chuyển nên giá trị cuối cùng chính là đỉnh mới.

## 5. Cách 2 — Hai queue, push `O(n)`, pop `O(1)` (hướng B)

Giữ **invariant**: `q1` luôn sắp theo thứ tự **stack**, tức **đầu `q1` = đỉnh stack**.

`push(x)`:
1. Đặt `x` vào `q2` rỗng → `x` đứng đầu `q2`.
2. Chuyển toàn bộ `q1` sang sau `x` trong `q2` (thứ tự cũ được giữ nguyên phía sau).
3. Đổi vai `q1, q2 = q2, q1` → `q1` giờ có `x` ở đầu, `q2` rỗng.

`pop()` = `q1.popleft()`; `top()` = `q1[0]`; `empty()` = `not q1`.

In [5]:
class MyStackTwoQueues:
    def __init__(self):
        self.q1 = deque()   # đầu q1 luôn là đỉnh stack
        self.q2 = deque()   # queue phụ, luôn rỗng giữa các lời gọi

    def push(self, x: int) -> None:
        self.q2.append(x)
        while self.q1:
            self.q2.append(self.q1.popleft())
        self.q1, self.q2 = self.q2, self.q1

    def pop(self) -> int:
        return self.q1.popleft()

    def top(self) -> int:
        return self.q1[0]

    def empty(self) -> bool:
        return not self.q1

s = MyStackTwoQueues()
for x in [1, 2, 3]:
    s.push(x)
    print(f"push {x}: q1 = {list(s.q1)} (đầu = đỉnh), q2 = {list(s.q2)}")
print("top =", s.top(), "| pop =", s.pop(), "| q1 =", list(s.q1))

push 1: q1 = [1] (đầu = đỉnh), q2 = []
push 2: q1 = [2, 1] (đầu = đỉnh), q2 = []
push 3: q1 = [3, 2, 1] (đầu = đỉnh), q2 = []
top = 3 | pop = 3 | q1 = [2, 1]


### 5.1. `q1, q2 = q2, q1` — đổi vai, không chép dữ liệu

Python chỉ đổi **hai cái tên** đang trỏ vào hai đối tượng deque; không có phần tử nào bị sao chép → `O(1)`.

In [6]:
a = deque([1, 2]); c = deque()
id_a, id_c = id(a), id(c)
a, c = c, a
print("a =", a, "| c =", c, "| a là đối tượng c cũ?", id(a) == id_c, "| c là đối tượng a cũ?", id(c) == id_a)

a = deque([]) | c = deque([1, 2]) | a là đối tượng c cũ? True | c là đối tượng a cũ? True


## 6. Insight cho follow-up — **một queue xoay vòng**

Ở cách 2, `q2` chỉ là "chỗ tạm" để xếp `x` lên trước. Có thể dùng chính `q` làm chỗ tạm:

1. `q.append(x)` → `x` nằm ở **cuối**.
2. Lặp `len(q) - 1` lần: `q.append(q.popleft())` — lấy người đầu hàng đưa xuống cuối hàng.
3. Sau `n - 1` lần xoay, mọi phần tử cũ đã đi ra sau `x` → `x` ở **đầu**.

```text
q = [3, 2, 1]  (đầu = 3 = đỉnh)     push 4:
append 4      -> [3, 2, 1, 4]
xoay 1 (3)    -> [2, 1, 4, 3]
xoay 2 (2)    -> [1, 4, 3, 2]
xoay 3 (1)    -> [4, 3, 2, 1]       đầu = 4 = đỉnh mới  ✓
```

Chỉ dùng `append`, `popleft`, `len` — đúng luật queue.

## 7. Cách 3 — Lời giải chính: một queue

Từng method:

**`__init__`** — một deque rỗng: `self.q = deque()`.

**`push(x)`** — thêm rồi xoay `len - 1` lần:
```python
self.q.append(x)
for _ in range(len(self.q) - 1):
    self.q.append(self.q.popleft())
```
`_` là quy ước đặt tên cho biến vòng lặp **không dùng tới**.

**`pop()`** — `return self.q.popleft()`.

**`top()`** — `return self.q[0]`.

**`empty()`** — `return not self.q`.

In [7]:
class MyStackOneQueue:
    def __init__(self):
        self.q = deque()

    def push(self, x: int) -> None:
        self.q.append(x)
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())

    def pop(self) -> int:
        return self.q.popleft()

    def top(self) -> int:
        return self.q[0]

    def empty(self) -> bool:
        return not self.q

s = MyStackOneQueue()
s.push(1); s.push(2)
print("top =", s.top(), "| pop =", s.pop(), "| empty =", s.empty(), "| pop =", s.pop(), "| empty =", s.empty())

top = 2 | pop = 2 | empty = False | pop = 1 | empty = True


## 8. TRACE

### 8.1. Bảng trace một queue: `push 1, push 2, push 3, top, pop, push 4, pop`

| Lệnh | Queue sau lệnh (đầu → cuối) | Trả về | Ghi chú |
|---|---|:-:|---|
| push 1 | [1] | — | xoay 0 lần |
| push 2 | [2, 1] | — | append → [1, 2], xoay 1 lần |
| push 3 | [3, 2, 1] | — | append → [2, 1, 3], xoay 2 lần |
| top | [3, 2, 1] | 3 | xem đầu |
| pop | [2, 1] | 3 | lấy đầu |
| push 4 | [4, 2, 1] | — | append → [2, 1, 4], xoay 2 lần |
| pop | [2, 1] | 4 | LIFO: 4 vào sau cùng, ra trước |

In [8]:
class TracedStack(MyStackOneQueue):
    # Kế thừa (inheritance): dùng lại mọi method của MyStackOneQueue, chỉ thêm print
    def push(self, x: int) -> None:
        self.q.append(x)
        print(f"push {x}: append -> {list(self.q)}")
        for r in range(len(self.q) - 1):
            moved = self.q.popleft()
            self.q.append(moved)
            print(f"    xoay {r + 1}: đưa {moved} xuống cuối -> {list(self.q)}")

def run_and_print(stack, ops):
    for op, *args in ops:
        result = getattr(stack, op)(*args)
        if op != "push":
            print(f"{op}() -> {result} | queue = {list(stack.q)}")

run_and_print(TracedStack(), [("push", 1), ("push", 2), ("push", 3), ("top",), ("pop",), ("push", 4), ("pop",)])

push 1: append -> [1]
push 2: append -> [1, 2]
    xoay 1: đưa 1 xuống cuối -> [2, 1]
push 3: append -> [2, 1, 3]
    xoay 1: đưa 2 xuống cuối -> [1, 3, 2]
    xoay 2: đưa 1 xuống cuối -> [3, 2, 1]
top() -> 3 | queue = [3, 2, 1]
pop() -> 3 | queue = [2, 1]
push 4: append -> [2, 1, 4]
    xoay 1: đưa 2 xuống cuối -> [1, 4, 2]
    xoay 2: đưa 1 xuống cuối -> [4, 2, 1]
pop() -> 4 | queue = [2, 1]


Hai cú pháp mới trong cell trên:
- `class TracedStack(MyStackOneQueue)` — **kế thừa**: class con có sẵn mọi method của class cha, ta chỉ **ghi đè** `push` để in thêm.
- `getattr(stack, "pop")` lấy method theo **tên dạng chuỗi**; `op, *args = ("push", 1)` gán `op = "push"`, `args = [1]` (dấu `*` gom phần còn lại thành list). Cách này cũng là cách ta mô phỏng trình chấm của LeetCode ở mục 9.

In [9]:
op, *args = ("push", 7)
print("op =", op, "| args =", args)
op, *args = ("top",)
print("op =", op, "| args =", args)
s = MyStackOneQueue()
getattr(s, "push")(5)
print("getattr(s, 'top')() =", getattr(s, "top")())

op = push | args = [7]
op = top | args = []
getattr(s, 'top')() = 5


### 8.2. Trace cách 1 (pop `O(n)`) để so sánh

Ở cách 1 queue lưu theo thứ tự **vào**, việc nặng dồn vào `pop`.

In [10]:
s = MyStackTwoQueuesLazy()
for x in [1, 2, 3]:
    s.push(x)
print("Sau 3 push: q1 =", list(s.q1), "| q2 =", list(s.q2), "| _top =", s._top)
print("pop ->", s.pop(), "| q1 =", list(s.q1), "| q2 =", list(s.q2), "| _top =", s._top)
print("pop ->", s.pop(), "| q1 =", list(s.q1), "| q2 =", list(s.q2), "| _top =", s._top)

Sau 3 push: q1 = [1, 2, 3] | q2 = [] | _top = 3
pop -> 3 | q1 = [1, 2] | q2 = [] | _top = 2
pop -> 2 | q1 = [1] | q2 = [] | _top = 1


## 9. Mô phỏng trình chấm LeetCode

LeetCode cho hai list: tên lệnh và tham số. Phần tử đầu `"MyStack"` là tạo đối tượng (trả `null`); các lệnh sau gọi method tương ứng. Kết quả `None` của Python hiển thị là `null`.

In [11]:
def run_leetcode(cls, commands, params):
    out = []
    obj = None
    for cmd, p in zip(commands, params):
        if cmd == cls.__name__ or cmd == "MyStack":
            obj = cls()
            out.append(None)
        else:
            out.append(getattr(obj, cmd)(*p))
    return out

commands = ["MyStack", "push", "push", "top", "pop", "empty"]
params = [[], [1], [2], [], [], []]
for cls in [MyStackTwoQueuesLazy, MyStackTwoQueues, MyStackOneQueue]:
    print(f"{cls.__name__:<22} -> {run_leetcode(cls, commands, params)}")

MyStackTwoQueuesLazy   -> [None, None, None, 2, 2, False]
MyStackTwoQueues       -> [None, None, None, 2, 2, False]
MyStackOneQueue        -> [None, None, None, 2, 2, False]


## 10. Code nộp LeetCode

Bản một queue (trả lời luôn follow-up). Trên LeetCode cần `from collections import deque` (thường đã được import sẵn, nhưng ghi rõ cho chắc).

In [12]:
from collections import deque

class MyStack:

    def __init__(self):
        self.q = deque()

    def push(self, x: int) -> None:
        self.q.append(x)
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())

    def pop(self) -> int:
        return self.q.popleft()

    def top(self) -> int:
        return self.q[0]

    def empty(self) -> bool:
        return not self.q

obj = MyStack()
obj.push(1); obj.push(2)
print(obj.top(), obj.pop(), obj.empty())

2 2 False


## 11. Vì sao đúng? (invariant)

**Cách 3** — sau mỗi lời gọi method:

> Đọc queue từ **đầu → cuối** cho đúng các phần tử của stack từ **đỉnh → đáy**.

- **Ban đầu:** rỗng ↔ rỗng → đúng.
- **`push(x)`:** trước đó queue = `[t1, t2, ..., tk]` (t1 là đỉnh). `append` cho `[t1, ..., tk, x]`. Mỗi lần xoay đưa phần tử đầu xuống cuối, sau `k` lần: `[x, t1, ..., tk]` — `x` ở đầu, thứ tự cũ giữ nguyên → invariant đúng với stack mới (đỉnh `x`).
- **`pop()`:** `popleft` bỏ `t1` = đỉnh, phần còn lại `[t2, ..., tk]` vẫn đúng thứ tự đỉnh → đáy.
- **`top()` / `empty()`:** chỉ đọc, không đổi trạng thái; `q[0]` = đỉnh, `not q` ⇔ stack rỗng.

Mọi thao tác chỉ dùng `append`, `popleft`, `q[0]`, `len`, `not q` — đúng luật queue.

## 12. Chi phí từng thao tác — bảng so sánh

Gọi `n` là số phần tử đang có trong stack.

| Cách | push | pop | top | empty | Số queue |
|---|---|---|---|---|:-:|
| 1. Hai queue, lười khi push | `O(1)` | `O(n)` | `O(1)` (nhờ `_top`) | `O(1)` | 2 |
| 2. Hai queue, chăm khi push | `O(n)` | `O(1)` | `O(1)` | `O(1)` | 2 |
| 3. Một queue xoay vòng | `O(n)` | `O(1)` | `O(1)` | `O(1)` | 1 |

Space: `O(n)` cho mọi cách.

**Chọn cách nào?** Nếu ứng dụng `pop`/`top` nhiều hơn `push` → cách 2/3. Nếu `push` nhiều mà `pop` ít → cách 1. Với đề này (≤ 100 lời gọi) mọi cách đều nhanh; cách 3 gọn nhất và trả lời follow-up.

In [13]:
# Đếm số thao tác queue cơ bản (append/popleft) để thấy chi phí thực tế
class CountingDeque(deque):
    ops = 0
    def append(self, x):
        CountingDeque.ops += 1
        super().append(x)
    def popleft(self):
        CountingDeque.ops += 1
        return super().popleft()

class OneQueueCounted(MyStackOneQueue):
    def __init__(self):
        self.q = CountingDeque()

for n in [10, 100, 1000]:
    CountingDeque.ops = 0
    st = OneQueueCounted()
    for i in range(n):
        st.push(i)
    print(f"{n} lần push (một queue): {CountingDeque.ops} thao tác cơ bản | n² = {n * n}")

10 lần push (một queue): 100 thao tác cơ bản | n² = 100
100 lần push (một queue): 10000 thao tác cơ bản | n² = 10000


1000 lần push (một queue): 1000000 thao tác cơ bản | n² = 1000000


Kết quả cho thấy `n` lần push tốn **đúng** `n²` thao tác cơ bản: push thứ `i` tốn `1` append + `(i - 1)` lần xoay × 2 thao tác = `2i - 1`, và `1 + 3 + 5 + ... + (2n - 1) = n²`. Với constraints ≤ 100 lời gọi, hoàn toàn chấp nhận được.

## 13. Bộ test

Kiểm thử bằng cách **so với stack thật** (`list` với `append`/`pop`) trên chuỗi lệnh ngẫu nhiên có seed: mọi giá trị trả về phải khớp.

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("ví dụ đề", run_leetcode(MyStack, commands, params), [None, None, None, 2, 2, False])
s = MyStack()
check("stack mới rỗng", s.empty(), True)
s.push(9)
check("1 phần tử: top", s.top(), 9)
check("1 phần tử: pop", s.pop(), 9)
check("rỗng lại", s.empty(), True)
for x in [1, 2, 3, 4, 5]:
    s.push(x)
check("LIFO 5 phần tử", [s.pop() for _ in range(5)], [5, 4, 3, 2, 1])
s.push(1); s.push(1)
check("giá trị trùng", (s.pop(), s.top()), (1, 1))

PASS ví dụ đề: [None, None, None, 2, 2, False]
PASS stack mới rỗng: True
PASS 1 phần tử: top: 9
PASS 1 phần tử: pop: 9
PASS rỗng lại: True
PASS LIFO 5 phần tử: [5, 4, 3, 2, 1]
PASS giá trị trùng: (1, 1)


In [15]:
import random
rng = random.Random(225)
for cls in [MyStackTwoQueuesLazy, MyStackTwoQueues, MyStackOneQueue, MyStack]:
    for _ in range(300):
        real, mine = [], cls()
        for _ in range(100):
            op = rng.choice(["push", "push", "pop", "top", "empty"])
            if op in ("pop", "top") and not real:
                op = "push"                         # đề đảm bảo pop/top luôn hợp lệ
            if op == "push":
                x = rng.randint(1, 9)
                real.append(x); mine.push(x)
            elif op == "pop":
                assert mine.pop() == real.pop()
            elif op == "top":
                assert mine.top() == real[-1]
            else:
                assert mine.empty() == (len(real) == 0)
    print(f"{cls.__name__:<22} khớp stack thật trên 300 chuỗi × 100 lệnh")

MyStackTwoQueuesLazy   khớp stack thật trên 300 chuỗi × 100 lệnh


MyStackTwoQueues       khớp stack thật trên 300 chuỗi × 100 lệnh
MyStackOneQueue        khớp stack thật trên 300 chuỗi × 100 lệnh
MyStack                khớp stack thật trên 300 chuỗi × 100 lệnh


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Xoay `len(q)` lần thay vì `len(q) - 1`

Xoay đủ `len(q)` lần thì queue quay về đúng thứ tự trước khi xoay → `x` lại nằm cuối.

In [16]:
class BugRotateAll(MyStackOneQueue):
    def push(self, x):
        self.q.append(x)
        for _ in range(len(self.q)):          # SAI: xoay đủ một vòng
            self.q.append(self.q.popleft())

s = BugRotateAll()
s.push(1); s.push(2); s.push(3)
print("queue:", list(s.q), "| top() =", s.top(), "(đúng phải là 3)")

queue: [1, 2, 3] | top() = 1 (đúng phải là 3)


### Lỗi 2 — Dùng thao tác không phải của queue

```python
def pop(self):
    return self.q.pop()        # lấy ở CUỐI -> đây là thao tác stack, vi phạm luật đề
def top(self):
    return self.q[-1]          # xem CUỐI -> cũng vi phạm
```
Code vẫn "Accepted" trên LeetCode (trình chấm không kiểm tra), nhưng **sai yêu cầu bài** và sẽ bị trừ điểm trong phỏng vấn.

### Lỗi 3 — Dùng `list.pop(0)` cho queue lớn

`list.pop(0)` là `O(n)` vì phải dịch mọi phần tử. Hợp lệ về luật nhưng chậm; dùng `deque.popleft()`.

In [17]:
import time
big_list = list(range(100_000)); big_deque = deque(range(100_000))
t0 = time.perf_counter()
for _ in range(20_000): big_list.pop(0)
t1 = time.perf_counter()
for _ in range(20_000): big_deque.popleft()
t2 = time.perf_counter()
print(f"20k lần list.pop(0): {t1 - t0:.4f}s | 20k lần deque.popleft(): {t2 - t1:.4f}s")

20k lần list.pop(0): 0.3252s | 20k lần deque.popleft(): 0.0010s


### Lỗi 4 — Cách 2 quên đổi vai `q1, q2`

Sau khi chuyển hết sang `q2`, nếu không đổi vai thì `q1` rỗng, `pop()` báo `IndexError: pop from an empty deque`.

### Lỗi 5 — Khai báo queue ở mức class thay vì trong `__init__`

```python
class MyStack:
    q = deque()        # SAI: mọi đối tượng MyStack dùng CHUNG một queue
```
LeetCode tạo nhiều đối tượng qua các test → dữ liệu test trước "rò" sang test sau.

In [18]:
class BugSharedQueue:
    q = deque()                     # thuộc tính lớp, dùng chung
    def push(self, x):
        self.q.append(x)
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())
    def empty(self):
        return not self.q

a1 = BugSharedQueue(); a1.push(5)
a2 = BugSharedQueue()
print("Stack mới a2.empty() =", a2.empty(), "(đúng phải là True) — vì a1 và a2 dùng chung q")

Stack mới a2.empty() = False (đúng phải là True) — vì a1 và a2 dùng chung q


## 15. Mẹo nhận dạng pattern

- Bài "Implement X using Y" là bài **thiết kế**: xác định **invariant** về cách dữ liệu được sắp trong Y, rồi mỗi method giữ invariant đó.
- Câu hỏi luôn là **"đẩy chi phí vào đâu?"** — vào thao tác ghi (push) hay thao tác đọc (pop/top).
- Đổi thứ tự FIFO ↔ LIFO: với queue → **xoay vòng**; với stack → **đổ sang stack thứ hai** (bài 232).
- Bài liên quan: 232 Implement Queue using Stacks (chi phí **khấu hao** `O(1)`), 155 Min Stack, 641 Design Circular Deque, 622 Design Circular Queue.

## 16. Biến thể — Implement Queue using Stacks (LeetCode 232)

Ngược lại bài này: dùng hai stack `inbox`, `outbox`.
- `push`: đẩy vào `inbox`.
- `pop`/`peek`: nếu `outbox` rỗng thì đổ **toàn bộ** `inbox` sang `outbox` (thứ tự bị đảo → phần tử cũ nhất lên đỉnh `outbox`), rồi lấy đỉnh `outbox`.

Mỗi phần tử bị chuyển **tối đa một lần** → chi phí **khấu hao** (amortized) `O(1)` mỗi thao tác. Đây là điểm hay mà bài stack-bằng-queue không có.

In [19]:
class MyQueue:
    def __init__(self):
        self.inbox, self.outbox = [], []
    def push(self, x):
        self.inbox.append(x)
    def _move(self):
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())
    def pop(self):
        self._move(); return self.outbox.pop()
    def peek(self):
        self._move(); return self.outbox[-1]
    def empty(self):
        return not self.inbox and not self.outbox

mq = MyQueue()
for x in [1, 2, 3]: mq.push(x)
print("FIFO:", mq.pop(), mq.peek(), mq.pop(), mq.pop(), "| empty:", mq.empty())

FIFO: 1 2 2 3 | empty: True


## 17. Bài tập tự luyện

### Bài 1 — Tính tay
Với cách một queue, viết trạng thái queue sau: `push 5, push 6, pop, push 7, push 8, top`. `top` trả gì?

### Bài 2 — Đếm
Cách một queue: tổng số lần gọi `popleft` khi push liên tiếp `n` phần tử vào stack rỗng là bao nhiêu (theo `n`)?

### Bài 3 — TODO: tự viết `push` của cách một queue

In [20]:
class ExerciseOneQueue:
    def __init__(self):
        self.q = deque()

    def push(self, x: int) -> None:
        # TODO: append x rồi xoay len(q) - 1 lần
        pass

    def pop(self) -> int:
        return self.q.popleft()

    def top(self) -> int:
        return self.q[0]

    def empty(self) -> bool:
        return not self.q

# s = ExerciseOneQueue(); s.push(1); s.push(2)
# assert s.top() == 2 and s.pop() == 2 and s.top() == 1

### Bài 4 — TODO: viết cách "xoay khi pop" bằng một queue
`push` chỉ `append`; `pop` xoay `len - 1` lần rồi `popleft`; `top` có thể làm như pop rồi đẩy lại xuống cuối.

In [21]:
class ExerciseRotateOnPop:
    def __init__(self):
        self.q = deque()

    def push(self, x: int) -> None:
        # TODO
        pass

    def pop(self) -> int:
        # TODO: xoay len - 1 lần, rồi popleft
        pass

    def top(self) -> int:
        # TODO: lấy như pop rồi append lại
        pass

    def empty(self) -> bool:
        # TODO
        pass

# s = ExerciseRotateOnPop(); s.push(1); s.push(2); s.push(3)
# assert s.top() == 3 and s.pop() == 3 and s.pop() == 2 and not s.empty()

### Bài 5 — Biến thể: thêm method `size()` và `peek_bottom()` (phần tử ở **đáy** stack) cho `MyStack` một queue
Gợi ý: dựa vào invariant "đầu = đỉnh", đáy nằm ở đâu? Có được đọc `q[-1]` không? (Hãy tự đặt luật: chỉ dùng thao tác queue.)

In [22]:
class ExerciseStackPlus(MyStackOneQueue):
    def size(self) -> int:
        # TODO
        pass

    def peek_bottom(self) -> int:
        # TODO: chỉ dùng thao tác queue (xoay len - 1 lần để đáy lên đầu, đọc, rồi xoay tiếp 1 lần cho về như cũ)
        pass

# s = ExerciseStackPlus(); s.push(1); s.push(2); s.push(3)
# assert s.size() == 3 and s.peek_bottom() == 1 and s.top() == 3

### Bài 6 — Mini challenge: Min Stack bằng queue
Viết `MinStackQ` dùng một queue (chỉ thao tác queue), hỗ trợ `push`, `pop`, `top`, `get_min()` trong đó `get_min` là `O(1)`. Gợi ý: mỗi phần tử trong queue lưu cặp `(giá_trị, min_tính_tới_lúc_đó)`.

In [23]:
class ExerciseMinStackQ:
    def __init__(self):
        self.q = deque()

    def push(self, x: int) -> None:
        # TODO
        pass

    def pop(self) -> int:
        # TODO
        pass

    def top(self) -> int:
        # TODO
        pass

    def get_min(self) -> int:
        # TODO
        pass

# m = ExerciseMinStackQ(); m.push(3); m.push(1); m.push(2)
# assert m.get_min() == 1 and m.pop() == 2 and m.pop() == 1 and m.get_min() == 3

## Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đầu queue luôn là đỉnh | sau `push 6`: `[6, 5]` | `pop` → `[5]`, `push 7` → `[7, 5]`, `push 8` → `[8, 7, 5]` |
| 2 | push thứ `i` xoay `i - 1` lần | cộng `0 + 1 + ... + (n-1)` | `n(n-1)/2` |
| 3 | append rồi xoay | `for _ in range(len(self.q) - 1)` | `self.q.append(self.q.popleft())` |
| 4 | dồn việc vào pop | xoay `len - 1` lần thì đỉnh lên đầu | `top`: `x = self.pop(); self.q.append(x); return x` |
| 5 | đầu = đỉnh ⇒ cuối = đáy | xoay `len - 1` lần → đáy lên đầu | đọc `q[0]` rồi xoay thêm 1 lần |
| 6 | mỗi phần tử nhớ min | `cur_min = min(x, self.q[0][1]) if self.q else x` | push cặp `(x, cur_min)` rồi xoay |

## Đáp án tham khảo

**Bài 1:** `[5]` → `[6, 5]` → pop 6 → `[5]` → `[7, 5]` → `[8, 7, 5]` → `top = 8`.

**Bài 2:** push thứ `i` (khi đã có `i - 1` phần tử) xoay `i - 1` lần, mỗi lần 1 `popleft` → tổng `0 + 1 + ... + (n - 1) = n(n - 1)/2`.

In [24]:
class AnswerOneQueue(MyStackOneQueue):
    pass                      # chính là cách 3

class AnswerRotateOnPop:
    def __init__(self):
        self.q = deque()
    def push(self, x):
        self.q.append(x)
    def pop(self):
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())
        return self.q.popleft()
    def top(self):
        x = self.pop()
        self.q.append(x)       # đặt lại xuống cuối -> vẫn là phần tử "mới nhất"
        return x
    def empty(self):
        return not self.q

class AnswerStackPlus(MyStackOneQueue):
    def size(self):
        return len(self.q)
    def peek_bottom(self):
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())
        bottom = self.q[0]
        self.q.append(self.q.popleft())      # xoay thêm 1 lần -> về đúng thứ tự cũ
        return bottom

class AnswerMinStackQ:
    def __init__(self):
        self.q = deque()
    def push(self, x):
        cur_min = min(x, self.q[0][1]) if self.q else x
        self.q.append((x, cur_min))
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())
    def pop(self):
        return self.q.popleft()[0]
    def top(self):
        return self.q[0][0]
    def get_min(self):
        return self.q[0][1]

s = AnswerOneQueue()
s.push(5); s.push(6); s.pop(); s.push(7); s.push(8)
assert list(s.q) == [8, 7, 5] and s.top() == 8

CountingDeque.ops = 0
st = OneQueueCounted()
for i in range(50): st.push(i)
assert CountingDeque.ops == 50 + 2 * (50 * 49 // 2)   # 50 append + mỗi lần xoay 1 popleft + 1 append

s = AnswerRotateOnPop(); s.push(1); s.push(2); s.push(3)
assert s.top() == 3 and s.pop() == 3 and s.pop() == 2 and not s.empty()
s = AnswerStackPlus(); s.push(1); s.push(2); s.push(3)
assert s.size() == 3 and s.peek_bottom() == 1 and s.top() == 3 and list(s.q) == [3, 2, 1]
m = AnswerMinStackQ(); m.push(3); m.push(1); m.push(2)
assert m.get_min() == 1 and m.pop() == 2 and m.pop() == 1 and m.get_min() == 3
print("Đáp án tham khảo PASS toàn bộ!")

Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
from collections import deque

class MyStack:
    def __init__(self):
        self.q = deque()                       # đầu queue = đỉnh stack
    def push(self, x):
        self.q.append(x)
        for _ in range(len(self.q) - 1):       # xoay len - 1 lần
            self.q.append(self.q.popleft())
    def pop(self):   return self.q.popleft()
    def top(self):   return self.q[0]
    def empty(self): return not self.q
```

**Checklist:**
- [ ] Chỉ dùng `append`, `popleft`, `q[0]`, `len`, `not q`.
- [ ] Xoay đúng `len(q) - 1` lần.
- [ ] Queue tạo trong `__init__` (không dùng thuộc tính lớp).
- [ ] Tên class `MyStack`, tên method đúng: `push`, `pop`, `top`, `empty`.
- [ ] Nói được chi phí: push `O(n)`, các thao tác khác `O(1)`.